In [0]:
dbutils.widgets.removeAll()

In [0]:
import logging
from datetime import datetime
import os

# Create logs directory if it doesn't exist
log_dir = "/Workspace/Users/saythu000@gmail.com/Fact_Revenue-Gap/logs"
dbutils.fs.mkdirs(f"file:{log_dir}")

# Create log filename with timestamp
log_filename = f"{log_dir}/provider_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_filename.replace('file:', '')),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info("=" * 60)
logger.info("Provider Silver Layer Processing Started")
logger.info(f"Log file: {log_filename}")
logger.info("=" * 60)

In [0]:
try:
    dbutils.widgets.text("ClientContainer", "claimsprocessing", "Client Container / Catalog Name")
    client_container = dbutils.widgets.get("ClientContainer").strip()
except Exception:
    client_container = "claimsprocessing"

# Wrap catalog name in backticks for safety (e.g. `274`)
safe_catalog = f"`{client_container}`"

sourcePath = f"/Volumes/{client_container}/bronze/processed_parquet/provider"
silverProviderBridgeTable = f"{safe_catalog}.silver.provider_person_bridge"
silverProviderTable = f"{safe_catalog}.silver.provider"
providerSpecialtyDatasetPath = "global/OperationalData/RAQ/TepReference/dbo/ProviderSpecialtyDataset"
carePreciseTaxoPath = "global/processed/Mult/Data/CarePreciseTaxo/custom/CarePreciseTaxo"
credentialingPath = f"{safe_catalog}.silver.ref_credentialing"
haiReportingListPath = f"{safe_catalog}.silver.ref_hai_reporting"
hospitalAffiliationPath = f"{safe_catalog}.silver.ref_hospital_affiliation" 

print("="*60)
print("CONFIGURATION")
print("="*60)
print(f"Source Path: {sourcePath}")
print(f"Silver Person Bridge Table: {silverProviderBridgeTable}")
print(f"Silver Provider Table: {silverProviderTable}")
print("="*60)

logger.info("Configuration loaded:")
logger.info(f"  Source Path: {sourcePath}")
logger.info(f"  Silver Person Bridge Table: {silverProviderBridgeTable}")
logger.info(f"  Silver Provider Table: {silverProviderTable}")

In [0]:
def table_exists(tableToCheck):
    """Check if a table or path exists and can be read"""
    try:
        if '/' in tableToCheck:
            # It is a path
            spark.read.format("parquet").load(tableToCheck)
        else:
            # It is a registered table name
            spark.table(tableToCheck)
        logger.info(f"Data check: {tableToCheck} - Data exists")
        return True
    except Exception as e:
        logger.warning(f"Data check: {tableToCheck} - Data does not exist or is inaccessible: {str(e)}")
        return False

In [0]:
from pyspark.sql.functions import date_format, sha2, concat_ws, col, row_number
from pyspark.sql.window import Window

In [0]:
srcsql = f"""
WITH consolidateProvider1 as (
  SELECT 
    brdg.BISInternalPersonID, brdg.UniqueRecord, prov.ClientID, prov.FileID, prov.LoadDateTime, 
    prov.FileLayoutID, prov.FileLayoutDescription, prov.ProviderID, prov.LastName, prov.MiddleInitial, 
    prov.FirstName, 
    coalesce(prov.TaxonomyCode1, cpt1.Taxo) AS TaxonomyCode1, 
    coalesce(s1.Specialty, prov.HpSpecialtyCode1) AS HpSpecialtyCode1, 
    coalesce(right(concat('00', cast(s1.CMSSpecialtyCode as string)), 2), prov.ADVProviderSpecialtyCode1) AS ADVProviderSpecialtyCode1, 
    coalesce(prov.TaxonomyCode2, cpt2.Taxo) AS TaxonomyCode2, 
    coalesce(s2.Specialty, prov.HpSpecialtyCode2) AS HpSpecialtyCode2, 
    coalesce(right(concat('00', cast(s2.CMSSpecialtyCode as string)), 2), prov.ADVProviderSpecialtyCode2) AS ADVProviderSpecialtyCode2, 
    coalesce(prov.TaxonomyCode3, cpt3.Taxo) AS TaxonomyCode3, 
    coalesce(s3.Specialty, prov.HpSpecialtyCode3) AS HpSpecialtyCode3, 
    coalesce(right(concat('00', cast(s3.CMSSpecialtyCode as string)), 2), prov.ADVProviderSpecialtyCode3) AS ADVProviderSpecialtyCode3, 
    coalesce(prov.TaxonomyCode4, cpt4.Taxo) AS TaxonomyCode4, 
    coalesce(s4.Specialty, prov.HpSpecialtyCode4) AS HpSpecialtyCode4, 
    coalesce(right(concat('00', cast(s4.CMSSpecialtyCode as string)), 2), prov.ADVProviderSpecialtyCode4) AS ADVProviderSpecialtyCode4, 
    coalesce(prov.TaxonomyCode5, cpt5.Taxo) AS TaxonomyCode5, 
    coalesce(s5.Specialty, prov.HpSpecialtyCode5) AS HpSpecialtyCode5, 
    coalesce(right(concat('00', cast(s5.CMSSpecialtyCode as string)), 2), prov.ADVProviderSpecialtyCode5) AS ADVProviderSpecialtyCode5, 
    prov.NPI, prov.PrescribePrivilege, prov.DEA, prov.PayorID, 
    coalesce(cred.Contracted, prov.Contracted, 'N') AS Contracted, 
    coalesce(hai.ProviderHAI, prov.ProviderHAI, 'N') AS ProviderHAI, 
    coalesce(hosp.HospitalID, prov.HospitalID) AS HospitalID, 
    coalesce(prov.ExcludeFromProviderReporting, 'N') AS ExcludeFromProviderReporting, 
    prov.AltProvReporting1, prov.AltProvReporting2, prov.AltProvReporting3, 
    prov.AltProvReporting4, prov.AltProvReporting5, prov.AltProvReporting6, 
    prov.AltProvReporting7, prov.AltProvReporting8, prov.AltProvReporting9, 
    prov.AltProvReporting10, brdg.PMUP, brdg.IsCurrentPMUP
  FROM consolidateProvider prov
  INNER JOIN silverProviderBridge brdg 
    ON prov.UniqueRecord = brdg.UniqueRecord 
    AND prov.FileLayoutID = brdg.FileLayoutID 
    AND brdg.IsCurrentPMUP = 1
  LEFT JOIN {safe_catalog}.silver.ref_careprecise_taxonomy cpt1 ON prov.TaxonomyCode1 = cpt1.Taxo
  LEFT JOIN {safe_catalog}.silver.ref_provider_specialty s1 ON prov.TaxonomyCode1 = s1.TaxonomyCode
  LEFT JOIN {safe_catalog}.silver.ref_careprecise_taxonomy cpt2 ON prov.TaxonomyCode2 = cpt2.Taxo
  LEFT JOIN {safe_catalog}.silver.ref_provider_specialty s2 ON prov.TaxonomyCode2 = s2.TaxonomyCode
  LEFT JOIN {safe_catalog}.silver.ref_careprecise_taxonomy cpt3 ON prov.TaxonomyCode3 = cpt3.Taxo
  LEFT JOIN {safe_catalog}.silver.ref_provider_specialty s3 ON prov.TaxonomyCode3 = s3.TaxonomyCode
  LEFT JOIN {safe_catalog}.silver.ref_careprecise_taxonomy cpt4 ON prov.TaxonomyCode4 = cpt4.Taxo
  LEFT JOIN {safe_catalog}.silver.ref_provider_specialty s4 ON prov.TaxonomyCode4 = s4.TaxonomyCode
  LEFT JOIN {safe_catalog}.silver.ref_careprecise_taxonomy cpt5 ON prov.TaxonomyCode5 = cpt5.Taxo
  LEFT JOIN {safe_catalog}.silver.ref_provider_specialty s5 ON prov.TaxonomyCode5 = s5.TaxonomyCode
  LEFT JOIN {safe_catalog}.silver.ref_credentialing cred ON prov.ProviderID = cred.ProviderID
  LEFT JOIN {safe_catalog}.silver.ref_hai_reporting hai ON prov.ProviderID = hai.ProviderID
  LEFT JOIN {safe_catalog}.silver.ref_hospital_affiliation hosp ON prov.ProviderID = hosp.ProviderID
)
SELECT 
  BISInternalPersonID, UniqueRecord, ClientID, FileID, LoadDateTime, FileLayoutID, 
  FileLayoutDescription, ProviderID, LastName, MiddleInitial, FirstName, 
  TaxonomyCode1, HpSpecialtyCode1, ADVProviderSpecialtyCode1, 
  TaxonomyCode2, HpSpecialtyCode2, ADVProviderSpecialtyCode2, 
  TaxonomyCode3, HpSpecialtyCode3, ADVProviderSpecialtyCode3, 
  TaxonomyCode4, HpSpecialtyCode4, ADVProviderSpecialtyCode4, 
  TaxonomyCode5, HpSpecialtyCode5, ADVProviderSpecialtyCode5, 
  NPI, PrescribePrivilege, DEA, PayorID, Contracted, 
  ProviderHAI, HospitalID, ExcludeFromProviderReporting, 
  AltProvReporting1, AltProvReporting2, AltProvReporting3, 
  AltProvReporting4, AltProvReporting5, AltProvReporting6, 
  AltProvReporting7, AltProvReporting8, AltProvReporting9, 
  AltProvReporting10, PMUP, IsCurrentPMUP,
  sha2(concat(
    IfNull(BISInternalPersonID,""), "|", IfNull(UniqueRecord,""), "|", 
    IfNull(ClientID,""), "|", IfNull(CAST(FileID AS STRING),""), "|", 
    IfNull(CAST(LoadDateTime AS STRING),""), "|", IfNull(CAST(FileLayoutID AS STRING),""), "|", 
    IfNull(FileLayoutDescription,""), "|", IfNull(ProviderID,""), "|", 
    IfNull(LastName,""), "|", IfNull(MiddleInitial,""), "|", IfNull(FirstName,""), "|", 
    IfNull(TaxonomyCode1,""), "|", IfNull(HpSpecialtyCode1,""), "|", IfNull(ADVProviderSpecialtyCode1,""), "|", 
    IfNull(TaxonomyCode2,""), "|", IfNull(HpSpecialtyCode2,""), "|", IfNull(ADVProviderSpecialtyCode2,""), "|", 
    IfNull(TaxonomyCode3,""), "|", IfNull(HpSpecialtyCode3,""), "|", IfNull(ADVProviderSpecialtyCode3,""), "|", 
    IfNull(TaxonomyCode4,""), "|", IfNull(HpSpecialtyCode4,""), "|", IfNull(ADVProviderSpecialtyCode4,""), "|", 
    IfNull(TaxonomyCode5,""), "|", IfNull(HpSpecialtyCode5,""), "|", IfNull(ADVProviderSpecialtyCode5,""), "|", 
    IfNull(NPI,""), "|", IfNull(PrescribePrivilege,""), "|", 
    IfNull(DEA,""), "|", IfNull(PayorID,""), "|", IfNull(Contracted,""), "|", 
    IfNull(ProviderHAI,""), "|", IfNull(HospitalID,""), "|", IfNull(ExcludeFromProviderReporting,""), "|", 
    IfNull(AltProvReporting1,""), "|", IfNull(AltProvReporting2,""), "|", 
    IfNull(AltProvReporting3,""), "|", IfNull(AltProvReporting4,""), "|", 
    IfNull(AltProvReporting5,""), "|", IfNull(AltProvReporting6,""), "|", 
    IfNull(AltProvReporting7,""), "|", IfNull(AltProvReporting8,""), "|", 
    IfNull(AltProvReporting9,""), "|", IfNull(AltProvReporting10,""), "|", 
    IfNull(PMUP,""), "|", IfNull(CAST(IsCurrentPMUP AS STRING),"")
  ), 256) AS HashKey 
FROM consolidateProvider1
"""

In [0]:
logger.info("MAIN EXECUTION STARTED")
print(f"Source Path: {sourcePath}")
print(f"Person Bridge: {silverProviderBridgeTable}")

if table_exists(sourcePath) and table_exists(silverProviderBridgeTable):
    print("Loading Bronze Provider data...")
    dfconsolidateProvSrc = spark.read.format("parquet").load(sourcePath)
    
    print("Loading Person Bridge table...")
    dfsilverProvBrdg = spark.table(silverProviderBridgeTable)
    
    # Run row fingerprints
    windowPartition = Window.partitionBy(col("FILE_ID")).orderBy(col("RecordHash").desc())
    dfconsolidateProv = dfconsolidateProvSrc.distinct() \
        .withColumn("RecordHash", sha2(concat_ws("||", *dfconsolidateProvSrc.columns), 256)) \
        .withColumn("RowNumber", row_number().over(windowPartition)) \
        .withColumn("UniqueRecord", concat_ws("-", col("FILE_ID"), col("RowNumber"))) \
        .withColumn("ClientID", col("CLIENT_ID")) \
        .withColumn("FileID", col("FILE_ID")) \
        .withColumn("LoadDateTime", col("LOAD_DATETIME")) \
        .withColumn("FileLayoutID", col("FILE_LAYOUT_ID").cast("string")) \
        .withColumn("FileLayoutDescription", col("FILE_LAYOUT_DESCRIPTION"))
        
    dfconsolidateProv.createOrReplaceTempView("consolidateProvider")
    dfsilverProvBrdg.createOrReplaceTempView("silverProviderBridge")
    
    print("Joining Provider profiles with Person Bridge...")
    dfsrc = spark.sql(srcsql)
    
    recordCount = dfsrc.count()
    print(f"Found {recordCount} records to write")
    
    if recordCount > 0:
        print(f"Writing to Silver Provider Table: {silverProviderTable}")
        spark.sql(f"DROP TABLE IF EXISTS {silverProviderTable}")
        dfsrc.write.format("delta").mode("overwrite").saveAsTable(silverProviderTable)
        print("Silver Provider table created successfully!")
    else:
        print("No records found to write.")
else:
    print("Bronze source or Person Bridge table does not exist. Skipping execution.")
    logger.error("Source or Bridge table missing")